# Analyse de la dépense publique par secteur d'activité (2024–2026)

**Problématique :** La commande publique française bénéficie-t-elle de manière disproportionnée à certains secteurs économiques par rapport à leur poids réel dans le tissu productif ?

### Sources de données mobilisées
1. **Données essentielles de la commande publique (DECP)** : fichiers annuels consolidés (`2024`, `2025`, `2026`) recensant les marchés publics notifiés, leurs montants, durées, procédures et titulaires.
2. **Base Sirene des établissements (`StockEtablissement`, Insee)** : répertoire exhaustif permettant d'identifier le code d'activité principale (`NAF Rev. 2`), la taille salariale, le statut de siège social et la localisation géographique de chaque établissement titulaire.
3. **Données financières des entreprises (INPI / BCE)** : liasses fiscales et ratios financiers conservés pour l'approfondissement financier ultérieur.

In [ ]:
# ==============================================================================
# 1. IMPORTS, CONFIGURATION ET CHARGEMENT DES DONNÉES BRUTES (DECP)
# ==============================================================================
import gc
from typing import Optional

import fsspec
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq
import requests
import seaborn as sns

from lire_fichiers import charger_donnees

# Style graphique global
sns.set_theme(style="whitegrid")

# URLs des fichiers Parquet stockés sur MinIO (SSP Cloud)
# Séparer le code (GitHub) des données volumineuses (S3/MinIO) garantit la reproductibilité.
URL_SIRENE_ETAB = (
    "https://minio.lab.sspcloud.fr/glemesre/diffusion/"
    "stock-stocketablissement-parquet.parquet"
)
URL_LIASSES_INPI = (
    "https://minio.lab.sspcloud.fr/k1000ensae/diffusion/projet-2a/data/"
    "donnees_financieres_entreprises.parquet"
)
URL_RATIOS_INPI = (
    "https://minio.lab.sspcloud.fr/glemesre/diffusion/"
    "ratios_inpi_bce(1).parquet"
)

# Paramètres méthodologiques de filtrage des DECP
ANNEE_MIN = 2024          # Fenêtre d'étude retenue (écarte les anciens contrats mis à jour tardivement)
MONTANT_MIN_EUROS = 100   # Écarte les accords-cadres saisis à 1 € symbolique
MONTANT_MAX_EUROS = 5e8   # Écarte les erreurs manifestes de saisie (> 500 M€)

# Chargement des millésimes DECP via le module local `lire_fichiers.py`
donnees_decp = charger_donnees()

for nom_fichier, table in donnees_decp.items():
    print(f"{nom_fichier} : {table.shape[0]:,} lignes, {table.shape[1]} colonnes")

# Aperçu rapide du premier millésime chargé
premier_millesime = next(iter(donnees_decp))
donnees_decp[premier_millesime].head(3)

In [ ]:
# ==============================================================================
# 2. DIAGNOSTIC DE COMPLÉTUDE (DECP) ET INSPECTION DES SOURCES EXTERNES
# ==============================================================================

# --- 2.1. Bilan des valeurs manquantes sur l'ensemble des fichiers DECP ---
total_lignes = sum(len(table) for table in donnees_decp.values())
valeurs_valides = pd.concat(
    [table.notna().sum() for table in donnees_decp.values()],
    axis=1,
).sum(axis=1)

bilan_nan = pd.DataFrame({
    "nb_nan": (total_lignes - valeurs_valides).astype(int),
    "pourcentage_nan": ((total_lignes - valeurs_valides) / total_lignes * 100).round(2),
}).sort_values(by="nb_nan", ascending=False)

colonnes_vides = bilan_nan.index[bilan_nan["pourcentage_nan"] == 100].tolist()
print(f"Colonnes 100 % vides dans les DECP ({len(colonnes_vides)}) : {colonnes_vides}")
display(bilan_nan.head(15))


# --- 2.2. Fonction utilitaire d'inspection légère des fichiers Parquet distants ---
def inspecter_parquet(url: str, nom_source: str, n_lignes: int = 5) -> pd.DataFrame:
    """Lit uniquement les métadonnées et un premier lot de lignes d'un fichier Parquet
    distant sans charger l'intégralité de la table en mémoire vive (RAM)."""
    with fsspec.open(url, "rb") as f:
        pf = pq.ParquetFile(f)
        print(f"\n[{nom_source}] {pf.metadata.num_rows:,} lignes | {len(pf.schema_arrow.names)} colonnes")
        print("Colonnes :", pf.schema_arrow.names)
        return next(pf.iter_batches(batch_size=n_lignes)).to_pandas()


# Vérification des schémas (Sirene + tables financières INPI réservées pour la suite)
apercu_sirene = inspecter_parquet(URL_SIRENE_ETAB, "Sirene - StockEtablissement")
apercu_ratios = inspecter_parquet(URL_RATIOS_INPI, "Ratios financiers INPI / BCE")
apercu_liasses = inspecter_parquet(URL_LIASSES_INPI, "Liasses fiscales INPI", n_lignes=500)

# Décodage d'un échantillon de liasses fiscales (bilans complets 'C' vs simplifiés 'S')
apercu_liasses["liasse"] = apercu_liasses["liasse"].apply(dict)
liasses_publiques = apercu_liasses[apercu_liasses["liasse"].str.len() > 0]
print("\nRépartition des types de bilans non confidentiels (échantillon) :")
print(liasses_publiques["type_bilan"].value_counts())

del apercu_sirene, apercu_ratios, apercu_liasses, liasses_publiques
gc.collect()

In [ ]:
# ==============================================================================
# 3. CONSOLIDATION ET NETTOYAGE DES DONNÉES DE LA COMMANDE PUBLIQUE (DECP)
# ==============================================================================

df_decp_raw = pd.concat(donnees_decp.values(), ignore_index=True)

# Suppression des colonnes 100 % vides et standardisation des noms ('.' -> '_')
df_decp_raw = df_decp_raw.drop(columns=colonnes_vides, errors="ignore")
df_decp_raw.columns = df_decp_raw.columns.str.replace(".", "_", regex=False)


def nettoyer_decp(
    df: pd.DataFrame,
    annee_min: Optional[int] = ANNEE_MIN,
    montant_min: float = MONTANT_MIN_EUROS,
    montant_max: float = MONTANT_MAX_EUROS,
) -> pd.DataFrame:
    """Nettoie et normalise la table des marchés publics (DECP) :
    1. Sélectionne les variables utiles et convertit les types numériques/dates.
    2. Filtre sur la période d'étude (écarte les anciens contrats republiés tardivement).
    3. Éclate les groupements d'entreprises (1 ligne = 1 couple marché x titulaire)
       et divise le montant par le nombre de co-titulaires pour éviter le double comptage.
    4. Normalise les identifiants SIRET (14 chiffres) et extrait le SIREN (9 chiffres).
    5. Dédoublonne sur le couple (identifiant marché, SIRET).
    6. Écarte les montants aberrants et calcule un montant annualisé.
    """
    df = df.copy()

    # 1. Sélection des variables d'intérêt
    id_col = "uid" if "uid" in df.columns else "id"
    cols_cibles = [
        id_col, "id", "nature", "procedure", "codeCPV",
        "montant", "dureeMois", "dateNotification",
        "acheteur_id", "acheteur_nom",
        "lieuExecution_code", "lieuExecution_typeCode",
        "titulaires",
    ]
    cols_keep = list(dict.fromkeys([c for c in cols_cibles if c in df.columns]))
    df = df[cols_keep]

    # 2. Conversion des types et filtrage temporel
    df["montant"] = pd.to_numeric(df["montant"], errors="coerce")
    df["dureeMois"] = pd.to_numeric(df["dureeMois"], errors="coerce")
    df["dateNotification"] = pd.to_datetime(df["dateNotification"], errors="coerce")

    if annee_min is not None:
        avant_date = len(df)
        df = df[df["dateNotification"].dt.year >= annee_min].copy()
        print(f"Contrats antérieurs à {annee_min} écartés : {avant_date - len(df):,}")

    # 3. Gestion des titulaires et des groupements d'entreprises
    has_tit = df["titulaires"].apply(
        lambda x: isinstance(x, (list, np.ndarray)) and len(x) > 0
    )
    df = df[has_tit].copy()

    # Répartition équitable du montant entre les co-titulaires d'un groupement
    df["nb_titulaires"] = df["titulaires"].apply(len)
    df["montant_part"] = df["montant"] / df["nb_titulaires"]

    # Passage de la maille 'contrat' à la maille 'contrat x titulaire'
    df = df.explode("titulaires").reset_index(drop=True)

    # Aplatissement des dictionnaires titulaires (gère l'encapsulation {'titulaire': {...}})
    tit_records = [
        t["titulaire"] if isinstance(t, dict) and "titulaire" in t
        else (t if isinstance(t, dict) else {})
        for t in df["titulaires"]
    ]
    df_tit = pd.json_normalize(tit_records).rename(columns={
        "id": "titulaire_id",
        "typeIdentifiant": "titulaire_type_id",
        "denominationSociale": "titulaire_nom",
    })
    cols_tit = [
        c for c in ["titulaire_id", "titulaire_type_id", "titulaire_nom"]
        if c in df_tit.columns
    ]
    df = pd.concat([df.drop(columns=["titulaires"]), df_tit[cols_tit]], axis=1)

    # 4. Filtrage sur les SIRET français valides (14 chiffres) et extraction du SIREN (9 chiffres)
    df["titulaire_id"] = df["titulaire_id"].astype(str).str.strip()
    is_siret = (
        (df["titulaire_type_id"] == "SIRET")
        & df["titulaire_id"].str.match(r"^\d{14}$", na=False)
    )
    print(f"Part de lignes avec SIRET valide (14 chiffres) : {is_siret.mean():.1%}")

    df = df[is_siret].copy()
    df["siret"] = df["titulaire_id"]
    df["siren"] = df["siret"].str[:9]

    # 5. Suppression des doublons de publication pour un même couple (marché, SIRET)
    avant_dedup = len(df)
    df = df.drop_duplicates(subset=[id_col, "siret"]).copy()
    print(f"Doublons (marché, SIRET) supprimés : {avant_dedup - len(df):,}")

    # 6. Filtrage des montants extrêmes et calcul du montant annualisé
    avant_filt = len(df)
    df = df[
        (df["montant_part"] > montant_min) & (df["montant_part"] < montant_max)
    ].copy()
    print(
        f"Lignes écartées pour montant aberrant "
        f"(<= {montant_min} € ou >= {montant_max:,.0f} €) : {avant_filt - len(df):,}"
    )

    duree_valide = df["dureeMois"].where(df["dureeMois"] > 0, np.nan)
    df["montant_annualise"] = df["montant_part"] / duree_valide * 12

    # Extraction de la division CPV (2 premiers chiffres = grande famille d'achat)
    if "codeCPV" in df.columns:
        df["cpv_division"] = df["codeCPV"].astype(str).str.slice(0, 2)

    return df


df_decp = nettoyer_decp(df_decp_raw)
del df_decp_raw, donnees_decp
gc.collect()
print(f"\nDECP nettoyées prêtes pour la jointure : {len(df_decp):,} lignes")

In [ ]:
# ==============================================================================
# 4. CHARGEMENT SÉLECTIF ET NETTOYAGE DE SIRENE (StockEtablissement)
# ==============================================================================

def associer_section_naf(code_naf: pd.Series) -> pd.Series:
    """Regroupe les 732 sous-classes NAF Rev. 2 (ex: '41.20A') en 21 sections
    officielles (A à U) à partir des deux premiers chiffres (division NAF)."""
    div = pd.to_numeric(code_naf.astype(str).str.slice(0, 2), errors="coerce")

    conditions = [
        div.between(1, 3),
        div.between(5, 9),
        div.between(10, 33),
        div == 35,
        div.between(36, 39),
        div.between(41, 43),
        div.between(45, 47),
        div.between(49, 53),
        div.between(55, 56),
        div.between(58, 63),
        div.between(64, 66),
        div == 68,
        div.between(69, 75),
        div.between(77, 82),
        div == 84,
        div == 85,
        div.between(86, 88),
        div.between(90, 93),
        div.between(94, 96),
        div.between(97, 98),
        div == 99,
    ]
    labels = [
        "A - Agriculture, sylviculture et pêche",
        "B - Industries extractives",
        "C - Industrie manufacturière",
        "D - Électricité, gaz, vapeur et air conditionné",
        "E - Eau, assainissement et déchets",
        "F - Construction (BTP)",
        "G - Commerce et réparation auto",
        "H - Transports et entreposage",
        "I - Hébergement et restauration",
        "J - Information et communication",
        "K - Activités financières et d'assurance",
        "L - Activités immobilières",
        "M - Activités spécialisées, scientifiques et techniques",
        "N - Services administratifs et de soutien",
        "O - Administration publique",
        "P - Enseignement",
        "Q - Santé humaine et action sociale",
        "R - Arts, spectacles et loisirs",
        "S - Autres activités de services",
        "T - Activités des ménages employeurs",
        "U - Organisations extraterritoriales",
    ]
    return pd.Series(np.select(conditions, labels, default="Inconnu"), index=code_naf.index)


# Lecture uniquement des 7 colonnes utiles pour limiter l'empreinte mémoire
cols_siret_cibles = [
    "siret",
    "etablissementSiege",
    "activitePrincipaleEtablissement",
    "nomenclatureActivitePrincipaleEtablissement",
    "trancheEffectifsEtablissement",
    "codeCommuneEtablissement",
    "etatAdministratifEtablissement",
]

with fsspec.open(URL_SIRENE_ETAB, "rb") as f:
    pf = pq.ParquetFile(f)
    cols_dispo = [c for c in cols_siret_cibles if c in pf.schema_arrow.names]
    df_sirene = pf.read(columns=cols_dispo).to_pandas()

# Normalisation de la clé SIRET (14 chiffres) et garantie d'unicité avant jointure
df_sirene["siret"] = df_sirene["siret"].astype(str).str.strip().str.zfill(14)
df_sirene = df_sirene.drop_duplicates(subset=["siret"]).copy()

# Création des variables sectorielles (code APE, division à 2 chiffres, section A-U)
df_sirene["naf_code"] = df_sirene["activitePrincipaleEtablissement"].astype(str).str.strip()
df_sirene["naf_division"] = df_sirene["naf_code"].str.slice(0, 2)
df_sirene["naf_section"] = associer_section_naf(df_sirene["naf_code"])

# Extraction du département de l'établissement (3 chiffres pour DROM 97x/98x, 2 en métropole)
if "codeCommuneEtablissement" in df_sirene.columns:
    commune = df_sirene["codeCommuneEtablissement"].astype(str).str.strip()
    df_sirene["dep_titulaire"] = np.where(
        commune.str.startswith(("97", "98")),
        commune.str.slice(0, 3),
        commune.str.slice(0, 2),
    )

# Référentiel économique national : poids démographique de chaque secteur actif en France.
# Calculé avant de supprimer `df_sirene` de la RAM pour servir de dénominateur à l'indice.
sirene_actifs = df_sirene[df_sirene["etatAdministratifEtablissement"] == "A"].copy()
sirene_actifs["siren_tmp"] = sirene_actifs["siret"].str[:9]

ref_secteurs_sirene = (
    sirene_actifs.groupby("naf_section", as_index=False)
    .agg(
        nb_etablissements_actifs=("siret", "count"),
        nb_entreprises_actives=("siren_tmp", "nunique"),
    )
)
ref_secteurs_sirene["part_etablissements_sirene"] = (
    ref_secteurs_sirene["nb_etablissements_actifs"]
    / ref_secteurs_sirene["nb_etablissements_actifs"].sum()
)

del sirene_actifs
gc.collect()
print(f"Sirene nettoyé : {len(df_sirene):,} établissements uniques prêts pour la jointure")

In [ ]:
# ==============================================================================
# 5. JOINTURE SÉCURISÉE (DECP x SIRENE), CONTRÔLE QUALITÉ ET EXPORT
# ==============================================================================

# `how="left"` conserve tous les marchés publics ;
# `validate="many_to_one"` garantit mathématiquement l'absence de duplication de montants.
df_final = pd.merge(
    df_decp,
    df_sirene,
    on="siret",
    how="left",
    validate="many_to_one",
)

# Libération immédiate de la mémoire de la table Sirene complète (~44,5 M de lignes)
del df_decp, df_sirene
gc.collect()

# Diagnostic de couverture de la jointure
taux_lignes = df_final["naf_code"].notna().mean()
montant_tot = df_final["montant_part"].sum()
montant_ok = df_final.loc[df_final["naf_code"].notna(), "montant_part"].sum()

print(f"Table finale (marché x titulaire)      : {len(df_final):,} lignes, {df_final.shape[1]} colonnes")
print(f"Taux d'appariement Sirene (en lignes)  : {taux_lignes:.1%}")
print(f"Taux d'appariement Sirene (en montant) : {montant_ok / montant_tot:.1%}")

# Sauvegarde au format Parquet pour accélérer les sessions suivantes sans recharger Sirene
df_final.to_parquet("decp_sirene_clean.parquet", index=False)
ref_secteurs_sirene.to_parquet("ref_secteurs_sirene.parquet", index=False)

# Tableau de synthèse : poids financier dans les DECP vs poids économique dans Sirene
synthese = (
    df_final.groupby("naf_section", as_index=False)
    .agg(
        nb_contrats=("siret", "count"),
        nb_entreprises=("siren", "nunique"),
        montant_total_M=("montant_part", lambda s: s.sum() / 1e6),
        montant_median=("montant_part", "median"),
    )
    .merge(ref_secteurs_sirene, on="naf_section", how="left")
)

synthese["part_montant_decp"] = (
    synthese["montant_total_M"] / synthese["montant_total_M"].sum()
)
# Indice > 1 : le secteur capte davantage de dépense publique que son poids économique
synthese["indice_surrepresentation"] = (
    synthese["part_montant_decp"] / synthese["part_etablissements_sirene"]
)

display(synthese.sort_values("montant_total_M", ascending=False).head(10))
df_final.head()

### Dictionnaire des variables (`df_final`)

#### 1. Clés de jointure et acteurs
| Variable | Source | Description |
| :--- | :--- | :--- |
| `uid` / `id` | DECP | Identifiant unique du marché public. |
| `siret` | **Clé** | Numéro SIRET normalisé (14 chiffres) de l'établissement titulaire. |
| `siren` | **Créée** | Numéro SIREN (9 chiffres) identifiant l'entreprise mère. |
| `titulaire_id` | DECP | Identifiant brut déclaré pour le titulaire avant nettoyage. |
| `titulaire_type_id` | DECP | Type d'identifiant (filtré uniquement sur `"SIRET"`). |
| `titulaire_nom` | DECP | Raison sociale du titulaire saisie par l'acheteur. |
| `acheteur_id` | DECP | SIRET de l'organisme public acheteur. |
| `acheteur_nom` | DECP | Nom de l'organisme public acheteur (État, collectivité, hôpital…). |

#### 2. Variables financières et temporelles
| Variable | Source | Description |
| :--- | :--- | :--- |
| `montant` | DECP | Montant total brut du contrat en euros (avant partage entre co-titulaires). |
| `nb_titulaires` | **Créée** | Nombre d'entreprises groupées sur le marché (`1` si titulaire unique). |
| `montant_part` | **Créée** | **Variable clé** : part du montant attribuée à l'établissement (`montant / nb_titulaires`). |
| `dureeMois` | DECP | Durée d'exécution prévue du contrat en mois. |
| `montant_annualise` | **Créée** | Montant ramené sur 12 mois (`montant_part / dureeMois * 12`). |
| `dateNotification` | DECP | Date officielle de signature du marché. |

#### 3. Nature et localisation du marché
| Variable | Source | Description |
| :--- | :--- | :--- |
| `nature` | DECP | Type de contrat (`Marché`, `Accord-cadre`, `Concession`…). |
| `procedure` | DECP | Mode de passation (`Appel d'offres ouvert`, `Procédure adaptée`, `Sans publicité`…). |
| `codeCPV` | DECP | Code européen à 8 chiffres décrivant l'objet de l'achat. |
| `cpv_division` | **Créée** | 2 premiers chiffres du CPV (grande famille d'achat, ex. `45` = BTP). |
| `lieuExecution_code` | DECP | Code géographique du lieu d'exécution de la prestation. |
| `lieuExecution_typeCode` | DECP | Échelle géographique (`Département`, `Commune`, `Code postal`, `Région`). |

#### 4. Profil sectoriel et géographique du titulaire (Sirene)
| Variable | Source | Description |
| :--- | :--- | :--- |
| `naf_section` | **Créée** | **Variable sectorielle principale** : les 21 sections NAF (`A` à `U`) avec libellé. |
| `naf_division` | **Créée** | 2 premiers chiffres du code APE (88 divisions fines). |
| `naf_code` | **Créée** | Code APE/NAF nettoyé (ex. `41.20A`). |
| `activitePrincipaleEtablissement` | Sirene | Code APE/NAF brut de l'établissement. |
| `nomenclatureActivitePrincipaleEtablissement` | Sirene | Version de la nomenclature (permet de vérifier `NAFRev2`). |
| `trancheEffectifsEtablissement` | Sirene | Code Insee de la tranche d'effectifs salariés de l'établissement. |
| `etablissementSiege` | Sirene | `True` si le titulaire est le siège social, `False` si c'est une agence locale. |
| `codeCommuneEtablissement` | Sirene | Code commune Insee d'implantation de l'établissement. |
| `dep_titulaire` | **Créée** | Département de l'établissement (`01` à `95`, `2A`/`2B`, `971` à `976`). |
| `etatAdministratifEtablissement` | Sirene | Statut administratif dans Sirene (`A` = actif, `F` = fermé). |

## 6. Visualisations et analyse exploratoire

Les graphiques suivants explorent la répartition sectorielle, la structure des montants par contrat, le rôle des sièges sociaux selon la procédure de passation et la concentration géographique des titulaires.

In [ ]:
# ==============================================================================
# 6.1. VOLUME FINANCIER TOTAL ATTRIBUÉ PAR SECTION NAF (TOUS SECTEURS)
# ==============================================================================

# Sous-ensemble des lignes valides appariées dans Sirene avec montant positif
df_apparies = df_final[
    df_final["naf_code"].notna()
    & (df_final["naf_section"] != "Inconnu")
    & (df_final["montant_part"] > 0)
].copy()

# Agrégation des montants (en M€) et du nombre de marchés distincts par section NAF
totaux_secteurs = (
    df_apparies.groupby("naf_section", observed=True, as_index=False)
    .agg(
        montant_total=("montant_part", "sum"),
        nb_marches=("id", "nunique"),
    )
    .sort_values("montant_total", ascending=False)
)
totaux_secteurs["montant_millions"] = totaux_secteurs["montant_total"] / 1e6

# Graphique en barres horizontales avec hauteur adaptée au nombre de sections
fig_height = max(7, len(totaux_secteurs) * 0.45)
fig, ax = plt.subplots(figsize=(12, fig_height))

sns.barplot(
    data=totaux_secteurs,
    x="montant_millions",
    y="naf_section",
    hue="naf_section",
    palette="viridis",
    legend=False,
    ax=ax,
)

# Ajout des valeurs directement au bout de chaque barre pour lire les petits secteurs
max_val = totaux_secteurs["montant_millions"].max()
for container in ax.containers:
    ax.bar_label(container, fmt=" %.1f M€", padding=4, fontsize=9, color="#222222")

ax.set_xlim(0, max_val * 1.18)
ax.xaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f} M€"))
ax.set_xlabel("Montant total attribué (en millions d'euros)", fontsize=11, labelpad=10)
ax.set_ylabel("")
ax.set_title(
    "Montant total des marchés publics attribués par section NAF de l'établissement",
    fontsize=13,
    fontweight="bold",
    pad=14,
)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.2. DISTRIBUTION DES MONTANTS ANNUALISÉS (TOP 10 SECTIONS NAF)
# ==============================================================================

# Sélection des 10 sections les plus fréquentes en nombre de contrats
top_10_sections = df_apparies["naf_section"].value_counts().nlargest(10).index
df_boxplot = df_apparies[
    df_apparies["naf_section"].isin(top_10_sections)
    & (df_apparies["montant_annualise"] > 0)
]

plt.figure(figsize=(11, 6))
sns.boxplot(
    data=df_boxplot,
    x="montant_annualise",
    y="naf_section",
    hue="naf_section",
    order=top_10_sections,
    palette="crest",
    legend=False,
    showfliers=False,  # Masque les valeurs extrêmes pour garder les boîtes interquartiles lisibles
)

plt.xscale("log")
plt.xlabel("Montant annualisé (€, échelle logarithmique)")
plt.ylabel("")
plt.title("Distribution des montants annualisés par grande section NAF (Top 10)")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.3. PART DES SIÈGES SOCIAUX SELON LA PROCÉDURE DE PASSATION
# ==============================================================================

# Table de correspondance des codes Insee de tranches d'effectifs salariés
libelles_effectifs = {
    "NN": "0 salarié / Non renseigné",
    "00": "0 salarié",
    "01": "1-2 sal.",
    "02": "3-5 sal.",
    "03": "6-9 sal.",
    "11": "10-19 sal.",
    "12": "20-49 sal.",
    "21": "50-99 sal.",
    "22": "100-199 sal.",
    "31": "200-249 sal.",
    "32": "250-499 sal.",
    "41": "500-999 sal.",
    "42": "1000-1999 sal.",
    "51": "2000-4999 sal.",
    "52": "5000+ sal.",
}
df_apparies["taille_etab"] = df_apparies["trancheEffectifsEtablissement"].map(libelles_effectifs)

# Répartition en volume financier (%) entre agences locales (False) et sièges sociaux (True)
ct_volume = (
    pd.crosstab(
        df_apparies["procedure"],
        df_apparies["etablissementSiege"],
        values=df_apparies["montant_part"],
        aggfunc="sum",
        normalize="index",
    )
    * 100
)

ax = ct_volume.plot(
    kind="barh",
    stacked=True,
    figsize=(10, 5),
    color=["#4c72b0", "#dd8452"],
)
ax.set_xlabel("Part du montant total attribué (%)")
ax.set_ylabel("")
ax.legend(
    ["Agence secondaire (False)", "Siège social (True)"],
    title="Type d'établissement",
    loc="lower right",
)
ax.set_title("Part des montants captés par les sièges sociaux selon la procédure")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 6.4. CARTOGRAPHIE DÉPARTEMENTALE DES MONTANTS ATTRIBUÉS (PLOTLY)
# ==============================================================================

# Chargement du contour simplifié des départements français (GeoJSON)
GEOJSON_URL = (
    "https://raw.githubusercontent.com/gregoiredavid/france-geojson/"
    "master/departements-version-simplifiee.geojson"
)
departements_geo = requests.get(GEOJSON_URL, timeout=30).json()

# Agrégation par département d'implantation de l'établissement titulaire
stats_dep = (
    df_apparies.groupby("dep_titulaire", as_index=False)
    .agg(
        nb_marches=("id", "nunique"),
        montant_total=("montant_part", "sum"),
        montant_median=("montant_part", "median"),
    )
)
stats_dep["montant_millions"] = stats_dep["montant_total"] / 1e6

# Carte choroplèthe interactive
fig = px.choropleth(
    stats_dep,
    geojson=departements_geo,
    locations="dep_titulaire",
    featureidkey="properties.code",
    color="montant_millions",
    hover_name="dep_titulaire",
    hover_data={"nb_marches": True, "montant_millions": ":.1f"},
    color_continuous_scale="Viridis",
    labels={
        "montant_millions": "Montant attribué (M€)",
        "nb_marches": "Nombre de marchés",
    },
    title="Montant des marchés publics par département de l'établissement titulaire",
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()